In [1]:
import scanpy as sc
import os

VISIUM_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"

# numeric .h5ad files only
visium_files = sorted(
    f for f in os.listdir(VISIUM_DIR)
    if f.endswith(".h5ad") and f.replace(".h5ad","").isdigit()
)

len(visium_files), visium_files[:5]

(41, ['0.h5ad', '10.h5ad', '100.h5ad', '102.h5ad', '106.h5ad'])

In [2]:
for f in visium_files:
    adata = sc.read_h5ad(os.path.join(VISIUM_DIR, f))
    print(f, adata.shape)

0.h5ad (6658, 13598)
10.h5ad (6138, 13598)
100.h5ad (7194, 13598)
102.h5ad (6847, 13598)
106.h5ad (7088, 13598)
107.h5ad (7057, 13598)
11.h5ad (7353, 13598)
111.h5ad (7046, 13598)
116.h5ad (8238, 13598)
117.h5ad (4778, 13598)
119.h5ad (7191, 13598)
122.h5ad (5903, 13598)
127.h5ad (7173, 13598)
128.h5ad (6661, 13598)
13.h5ad (5910, 13598)
14.h5ad (6855, 13598)
18.h5ad (6319, 13598)
28.h5ad (6989, 13598)
29.h5ad (6768, 13598)
3.h5ad (5571, 13598)
30.h5ad (6476, 13598)
34.h5ad (6297, 13598)
36.h5ad (6873, 13598)
38.h5ad (6971, 13598)
40.h5ad (7217, 13598)
46.h5ad (6826, 13598)
49.h5ad (6502, 13598)
51.h5ad (6342, 13598)
52.h5ad (6718, 13598)
6.h5ad (6414, 13598)
60.h5ad (6492, 13598)
72.h5ad (6706, 13598)
73.h5ad (6532, 13598)
74.h5ad (7010, 13598)
80.h5ad (6526, 13598)
81.h5ad (7274, 13598)
83.h5ad (7485, 13598)
86.h5ad (7480, 13598)
91.h5ad (6651, 13598)
96.h5ad (6987, 13598)
97.h5ad (6142, 13598)


In [4]:
import scanpy as sc
import pandas as pd
import os

VISIUM_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
META_PATH  = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"

meta = pd.read_csv(META_PATH)
meta["deident"] = meta["deident"].astype(str)

def classify_mets(row):
    # row fields look like T/F and TRUE/FALSE depending on how pandas read them
    any_mets = str(row["any_mets"]).strip().upper() in ["T", "TRUE", "1"]
    ln_only  = str(row["ln_only"]).strip().upper() in ["T", "TRUE", "1"]
    dist     = str(row["Distant_Mets"]).strip().upper() in ["T", "TRUE", "1"]

    if not any_mets:
        return "No"
    if dist:
        return "Distant"
    # if any_mets but not distant, treat as LN (ln_only should usually be True here)
    return "LN"

# numeric .h5ad files only
visium_files = sorted(
    f for f in os.listdir(VISIUM_DIR)
    if f.endswith(".h5ad") and f.replace(".h5ad","").isdigit()
)

adata_list = []

for f in visium_files:
    sample_id = f.replace(".h5ad","")  # e.g. "127"
    adata = sc.read_h5ad(os.path.join(VISIUM_DIR, f))

    row = meta.loc[meta["deident"] == sample_id]
    if row.shape[0] != 1:
        raise ValueError(f"Expected 1 metadata row for deident={sample_id}, found {row.shape[0]}")
    row = row.iloc[0]

    adata.obs["patient_id"]  = sample_id
    adata.obs["mets_status"] = classify_mets(row)
    adata.obs["MLH1"]        = row["MLH1"]
    adata.obs["age"]         = row["age"]
    adata.obs["sex"]         = row["sex"]

    adata_list.append(adata)

# quick check
adata_list[0].obs[["patient_id","mets_status","MLH1","age","sex"]].head()

,patient_id,mets_status,MLH1,age,sex
AACAATCCGAGTGGAC-1,0,LN,1,62,M
AACAATGTGCTCCGAG-1,0,LN,1,62,M
AACACGACAACGGAGT-1,0,LN,1,62,M
AACACGACAATTGTTC-1,0,LN,1,62,M
AACACGGAACGAGTTA-1,0,LN,1,62,M


In [5]:
meta[["deident","any_mets","ln_only","Distant_Mets"]].head(10)

,deident,any_mets,ln_only,Distant_Mets
0,0,T,T,False
1,2,T,F,True
2,3,F,F,False
3,6,T,F,True
4,10,F,F,False
5,11,T,F,True
6,13,F,F,False
7,14,F,F,False
8,18,T,F,True
9,19,F,F,False


In [ ]:
Test one sample

In [6]:
# pick one sample to inspect
adata = adata_list[0]

# list all Cell2Location columns
abundance_cols = adata.obsm["means_cell_abundance_w_sf"].columns
abundance_cols[:10]

Index(['meanscell_abundance_w_sf_Tumor_cE01 (Stem/TA-like)',
       'meanscell_abundance_w_sf_Tumor_cE02 (Stem/TA-like/Immature Goblet)',
       'meanscell_abundance_w_sf_Tumor_cE03 (Stem/TA-like prolif)',
       'meanscell_abundance_w_sf_Tumor_cE04 (Enterocyte 1)',
       'meanscell_abundance_w_sf_Tumor_cE05 (Enterocyte 2)',
       'meanscell_abundance_w_sf_Tumor_cE06 (Immature Goblet)',
       'meanscell_abundance_w_sf_Tumor_cE07 (Goblet/Enterocyte)',
       'meanscell_abundance_w_sf_Tumor_cE08 (Goblet)',
       'meanscell_abundance_w_sf_Tumor_cE09 (Best4)',
       'meanscell_abundance_w_sf_Tumor_cE10 (Tuft)'],
      dtype='object')

In [7]:
dc2_col = "meanscell_abundance_w_sf_cM04"

# add DC2 abundance to obs
adata.obs["DC2_abundance"] = adata.obsm["means_cell_abundance_w_sf"][dc2_col]

adata.obs[["DC2_abundance"]].head()

KeyError: 'meanscell_abundance_w_sf_cM04'

In [8]:
[c for c in abundance_cols if "cM04" in c]

['meanscell_abundance_w_sf_cM04 (DC2)']

In [9]:
adata = adata_list[0]

abundance_cols = adata.obsm["means_cell_abundance_w_sf"].columns

dc_cols = [c for c in abundance_cols if "(DC" in c or "pDC" in c or "mregDC" in c]

dc_cols

['meanscell_abundance_w_sf_cM03 (DC1)',
 'meanscell_abundance_w_sf_cM04 (DC2)',
 'meanscell_abundance_w_sf_cM05 (DC2 C1Q+)',
 'meanscell_abundance_w_sf_cM06 (DC IL22RA2)',
 'meanscell_abundance_w_sf_cM07 (pDC)',
 'meanscell_abundance_w_sf_cM09 (mregDC)']

In [10]:
import pandas as pd

dc2_key = "cM04"  # DC2

rows = []

for adata in adata_list:
    ab = adata.obsm["means_cell_abundance_w_sf"]
    dc2_col = [c for c in ab.columns if dc2_key in c][0]   # exact column name

    obs = adata.obs.copy()

    # add DC2 abundance
    obs["DC2_abundance"] = ab[dc2_col].values

    # keep only columns we need (region column added in next line once we find it)
    keep = ["patient_id", "mets_status", "MLH1", "DC2_abundance"]
    keep = [k for k in keep if k in obs.columns]  # safe

    rows.append(obs[keep])

dc2_spots = pd.concat(rows, axis=0)

dc2_spots.shape, dc2_spots.head()

((275658, 4),
                    patient_id mets_status  MLH1  DC2_abundance
 AACAATCCGAGTGGAC-1          0          LN     1       0.211424
 AACAATGTGCTCCGAG-1          0          LN     1       0.041351
 AACACGACAACGGAGT-1          0          LN     1       0.041520
 AACACGACAATTGTTC-1          0          LN     1       0.082085
 AACACGGAACGAGTTA-1          0          LN     1       0.124227)

In [11]:
adata_list[0].obs.columns

Index(['in_tissue', 'array_row', 'array_col', '_indices', '_scvi_batch',
       '_scvi_labels', 'histology', 'patient_id', 'mets_status', 'MLH1', 'age',
       'sex'],
      dtype='object')

In [12]:
rows = []

for adata in adata_list:
    ab = adata.obsm["means_cell_abundance_w_sf"]
    dc2_col = [c for c in ab.columns if "cM04" in c][0]

    obs = adata.obs.copy()
    obs["DC2_abundance"] = ab[dc2_col].values

    keep = ["patient_id", "mets_status", "MLH1", "histology", "DC2_abundance"]
    rows.append(obs[keep])

dc2_spots = pd.concat(rows, axis=0)

dc2_spots.head()

,patient_id,mets_status,MLH1,histology,DC2_abundance
AACAATCCGAGTGGAC-1,0,LN,1,cancer_cancer,0.211424
AACAATGTGCTCCGAG-1,0,LN,1,Unannotated,0.041351
AACACGACAACGGAGT-1,0,LN,1,cancer_cancer,0.041520
AACACGACAATTGTTC-1,0,LN,1,cancer_cancer,0.082085
AACACGGAACGAGTTA-1,0,LN,1,cancer_cancer,0.124227


In [13]:
dc2_spots["histology"].value_counts()

histology
cancer_cancer          171669
benign_muscularis       22967
inter                   19630
benign_fat              18369
benign_stroma           14037
Unannotated              9699
benign_epithelium        8856
benign_submucosa         7163
benign_inflammation      2918
benign_serosa             350
Name: count, dtype: int64

In [14]:
dc2_spots.groupby("histology")["DC2_abundance"].describe()

,count,mean,std,min,25%,50%,75%,max
histology,,,,,,,,
Unannotated,9699.0,0.081505,0.074389,0.026001,0.044151,0.055969,0.081010,0.809041
benign_epithelium,8856.0,0.081543,0.066901,0.026294,0.049573,0.057413,0.082635,0.735700
benign_fat,18369.0,0.057466,0.044637,0.030770,0.038388,0.044378,0.056614,0.666395
benign_inflammation,2918.0,0.224066,0.106159,0.032398,0.143340,0.234441,0.278873,0.673633
benign_muscularis,22967.0,0.070437,0.048807,0.026707,0.047863,0.057241,0.072587,0.683953
benign_serosa,350.0,0.066142,0.052527,0.031060,0.044200,0.052195,0.063731,0.490140
benign_stroma,14037.0,0.086471,0.082510,0.031029,0.046403,0.058278,0.086562,0.734659
benign_submucosa,7163.0,0.068471,0.050572,0.031703,0.045025,0.055212,0.072364,0.735851
cancer_cancer,171669.0,0.098544,0.078442,0.024760,0.053278,0.072196,0.113615,0.994942


In [27]:
import numpy as np
from scipy.spatial import KDTree
from scipy.spatial.distance import pdist

def define_regions_spatial(adata):
    """
    Defines:
      - Center Tumor (CT)
      - Invasive Margin (IM)
      - Other
    using explicit physical distances in Visium space
    """

    # -------------------------------
    # 1. Spatial coordinates (pixels)
    # -------------------------------
    coords = adata.obsm["spatial"]

    # -------------------------------
    # 2. Estimate spot size (~100 µm)
    # -------------------------------
    # median nearest-neighbor distance ≈ one Visium spot
    pixel_scale_100um = np.median(pdist(coords[:200]))

    # -------------------------------
    # 3. Histology masks
    # -------------------------------
    tumor_labels  = ["cancer_cancer"]
    stroma_labels = ["benign_stroma", "benign_fat", "inter", "stroma"]

    histo = adata.obs["histology"].astype(str)
    is_tumor  = histo.isin(tumor_labels).values
    is_stroma = histo.isin(stroma_labels).values

    region_labels = np.array(["Other"] * adata.n_obs, dtype=object)

    # -------------------------------
    # 4. Find tumor–stroma interface
    # -------------------------------
    tumor_coords  = coords[is_tumor]
    stroma_coords = coords[is_stroma]

    if len(tumor_coords) == 0 or len(stroma_coords) == 0:
        return region_labels

    stroma_tree = KDTree(stroma_coords)

    # distance from each tumor spot to nearest stroma spot
    dists_to_stroma, _ = stroma_tree.query(tumor_coords, k=1)
    dists_to_stroma = dists_to_stroma.ravel()

    # -------------------------------
    # 5. Boundary tumor spots (~150 µm)
    # -------------------------------
    boundary_threshold = 1.5 * pixel_scale_100um   # ≈ 150 µm
    boundary_tumor_mask = dists_to_stroma <= boundary_threshold

    tumor_indices = np.where(is_tumor)[0]
    boundary_indices = tumor_indices[boundary_tumor_mask]

    if len(boundary_indices) == 0:
        region_labels[is_tumor] = "Center Tumor (CT)"
        return region_labels

    # -------------------------------
    # 6. Define invasive margin (~500 µm)
    # -------------------------------
    boundary_coords = coords[boundary_indices]
    boundary_tree = KDTree(boundary_coords)

    dists_to_boundary, _ = boundary_tree.query(coords, k=1)
    dists_to_boundary = dists_to_boundary.ravel()

    im_width = 5 * pixel_scale_100um   # ≈ 500 µm

    is_im = dists_to_boundary <= im_width

    # -------------------------------
    # 7. Assign regions
    # -------------------------------
    region_labels[is_im & (is_tumor | is_stroma)] = "Invasive Margin (IM)"
    region_labels[is_tumor & (~is_im)] = "Center Tumor (CT)"

    return region_labels

In [30]:
#Now add the region info
pixel_scale_100um = 1.0  # same assumption as before

for adata in adata_list:
    adata.obs["region"] = define_regions_spatial(adata)
    

In [31]:
adata = adata_list[0]
adata.obs["region"].value_counts()

region
Invasive Margin (IM)    6289
Other                    369
Name: count, dtype: int64